In [1]:
#r "nuget: ScottPlot, 5.0.21"
#r "C:/Users/эльмира/Desktop/litau/practice2026/practice2026/task14/bin/Debug/net9.0/task14.dll"

using System;
using System.IO;
using System.Diagnostics;
using System.Collections.Generic;
using System.Linq;
using ScottPlot;

double a = -100;
double b = 100;
Func<double, double> sinFunc = Math.Sin;
double step = 1e-5; 
int warmups = 3;    
int runs = 10;      

int[] threadCounts = Enumerable.Range(1, 16).ToArray();
double[] averageTimesMs = new double[threadCounts.Length];

// Разогрев JIT
DefiniteIntegral.Solve(a, b, sinFunc, step, 2);

Console.WriteLine("Начинаем замеры многопотока...");

for (int t = 0; t < threadCounts.Length; t++)
{
    int threads = threadCounts[t];
    
    // Прогреваем
    for (int i = 0; i < warmups; i++)
    {
        DefiniteIntegral.Solve(a, b, sinFunc, step, threads);
    }

    List<double> times = new List<double>();
    Stopwatch sw = new Stopwatch();

    for (int i = 0; i < runs; i++)
    {
        sw.Restart();
        DefiniteIntegral.Solve(a, b, sinFunc, step, threads);
        sw.Stop();
        times.Add(sw.Elapsed.TotalMilliseconds);
    }

    averageTimesMs[t] = times.Average();
    Console.WriteLine($"Потоков: {threads:D2} | Среднее время: {averageTimesMs[t]:F2} мс");
}

Console.WriteLine("\nЗамеряем чистый однопоток...");
List<double> singleThreadTimes = new List<double>();
Stopwatch swSingle = new Stopwatch();

for (int i = 0; i < runs; i++)
{
    swSingle.Restart();
    DefiniteIntegral.Solve(a, b, sinFunc, step, 1);
    swSingle.Stop();
    singleThreadTimes.Add(swSingle.Elapsed.TotalMilliseconds);
}
double avgSingleTime = singleThreadTimes.Average();
double minMultiTime = averageTimesMs.Min();
int bestThreadCount = threadCounts[Array.IndexOf(averageTimesMs, minMultiTime)];
double speedupPercent = ((avgSingleTime - minMultiTime) / avgSingleTime) * 100;

Console.WriteLine($"\n=== ИТОГ ===");
Console.WriteLine($"Однопоток: {avgSingleTime:F2} мс");
Console.WriteLine($"Лучший многопоток ({bestThreadCount} пот.): {minMultiTime:F2} мс");
Console.WriteLine($"Ускорение: {speedupPercent:F2}%");

var plt = new ScottPlot.Plot();
plt.Title("Зависимость времени выполнения Solve от количества потоков (Шаг 1e-5)");
plt.XLabel("Время выполнения (мс)");
plt.YLabel("Количество потоков");

double[] xs = averageTimesMs;
double[] ys = threadCounts.Select(t => (double)t).ToArray();

var scatter = plt.Add.Scatter(xs, ys);
scatter.LineWidth = 2.5f;
scatter.MarkerSize = 8.0f;

plt.SavePng("performance_chart.png", 800, 600);
Console.WriteLine("\nГрафик сохранен как 'performance_chart.png'!");

string resultsPath = "results.txt";
using (StreamWriter writer = new StreamWriter(resultsPath))
{
    writer.WriteLine($"Оптимальный размер шага: {step:E3}");
    writer.WriteLine($"Оптимальное количество потоков: {bestThreadCount}");
    writer.WriteLine($"Скорость однопоточной версии: {avgSingleTime:F2} мс");
    writer.WriteLine($"Скорость многопоточной версии: {minMultiTime:F2} мс");
    writer.WriteLine($"Ускорение: {speedupPercent:F2}%");
}
Console.WriteLine("Результаты успешно сохранены в файл 'results.txt'!");

Installed Packages ScottPlot, 5.0.21

Loading extensions from `C:\Users\эльмира\.nuget\packages\skiasharp\2.88.7\interactive-extensions\dotnet\SkiaSharp.DotNet.Interactive.dll`

-5.415112802609201E-14
Начинаем замеры многопотока...
-2.5129951356059506E-15
-2.5129951356059506E-15
-2.5129951356059506E-15
-2.5129951356059506E-15
-2.5129951356059506E-15
-2.5129951356059506E-15
-2.5129951356059506E-15
-2.5129951356059506E-15
-2.5129951356059506E-15
-2.5129951356059506E-15
-2.5129951356059506E-15
-2.5129951356059506E-15
-2.5129951356059506E-15
Потоков: 01 | Среднее время: 308.95 мс
-5.415112802609201E-14
-5.415112802609201E-14
-5.415112802609201E-14
-5.415112802609201E-14
-5.415112802609201E-14
-5.415112802609201E-14
-5.415112802609201E-14
-5.415112802609201E-14
-5.415112802609201E-14
-5.415112802609201E-14
-5.415112802609201E-14
-5.415112802609201E-14
-5.415112802609201E-14
Потоков: 02 | Среднее время: 153.58 мс
-1.545430450278218E-13
-1.545430450278218E-13
-1.545430450278218E-13
-1.545430450278218E-13
-1.545430450278218E-13
-1.545430450278218E-13
-1.545430450278218E-13
-1.5456792069141675E-13
-1.5456792069141675E-13
-1.545430450278218E-13
-1.545430450278218E-13
-1